In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A2_generation/9_7_ml_rnn.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Melody Generation with Machine Learning

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 21.05.2026
+ **Author:** Dr. Benedikt Zönnchen

In the following we will create music sheets and sound. For those tasks ``Python`` requires external programs that you should install if you are working locally:

1. [Musescore](https://musescore.org/de) (for generating sheets)
2. [FluidSynth](https://www.fluidsynth.org/) (for generating sound)

If you are working on google ``Colab``, you can evaluate the following three cells to install these applications:

In [ ]:
#@title install dependencies to play sound
%%capture
print('installing fluidsynth...')
!apt-get install fluidsynth > /dev/null
!cp /usr/share/sounds/sf2/FluidR3_GM.sf2 ./font.sf2
print('done!')

In [ ]:
#@title install dependencies to show score in music notation
%%capture
print('installing musescore3...')
!apt-get install musescore3 > /dev/null
print('done!')

Furtheremore, for this notebook we need the following ``Python`` packages and moduls. Execute the cell to install them:

In [ ]:
%pip install music21
%pip install pyfluidsynth

%pip install matplotlib
%pip install seaborn

%pip install pandas
%pip install numpy
%pip install torch

%pip install otter-grader==5.5.0

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import requests, os
    
    folders = ['tests', 'data', 'figs']
    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        api = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A2_generation/{folder}"
        for f in requests.get(api).json():
            open(f"tests/{f['name']}", 'w').write(requests.get(f['download_url']).text)
    
    for f in requests.get(link).json():
        if f['name'].endswith('.py'):
            open(f['name'], 'w').write(requests.get(f['download_url']).text)

    # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('9_7_ml_rnn.ipynb')

In [ ]:
import torch
import numpy as np
import music21 as m21

import zipfile
from files import load_midi_files
from pianoroll import stream_to_df, plot_df
from encoder import PianoRollEncoder, StringToIntEncoder, TERM_SYMBOL

## 24 Recurrent Neural Networks (RNN) and LSTM

### Mathematical Assumptions: Autoregression

Where in [23 Learning the Markov Matrix](https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A2_generation/9_6_ml_ffn_markov.ipynb) we assumed the Markov condition, that is, that any the probability of the next event occuring only depends on the last event, we know relax this strong and rather unreasonable assumption.

Instead we assume a sequence of random variables $X_1, X_2, \ldots, X_k$ that are autoregressively dependent, that is,

\begin{align*}
P(x_{k+1}, x_k, x_{k-1}, \ldots, x_{1}) = P(x_{k+1} | x_{k}, \ldots, x_{1}) \cdot P(x_{k} | x_{k-1}, \ldots, x_{1}) \cdot \cdots \cdot P(x_2 | x_1) \cdot P(x_1)
\end{align*}

Here we wrote $P(x_1)$ as a shorthand for $P(X_1 = x_1)$. In other words we assume that the next musical event / note depends on all the events happened / computed before!

---

🗣 **Hint:** All large language model work under this assumption. They are autoregressive models!

---


### RNNs

RNNs let us predict the next element of a sequence given a finite — but arbitrarily long — history of preceding elements. Therefore, they are a good model type to realize the autoregressive mathematical model. (How *accurate* that prediction is, is a separate question.) Unlike the model from the previous section, an RNN is **not memoryless** while executing: it maintains an internal state that summarizes everything it has seen so far.

Suppose we have a sequence of notes/events $\mathbf{x}_0 = $ ``C4``, $\mathbf{x}_1 = $ ``G4``, $\mathbf{x}_2 = $ ``_``, $\ldots$. We feed ``C4`` into the RNN. It produces both an output $\mathbf{y}_0$ — the predicted next note — and a **hidden state** $\mathbf{h}_1$ ($\mathbf{h}_0$ is just the empty state), which acts as memory. When we feed in the next note ``G4``, the previous hidden state $\mathbf{h}_1$ is also part of the input, and a new hidden state $\mathbf{h}_2$ is produced. The figure below shows an RNN *unfolded in time*.

<img src="figs/rnn-unfold.png" alt="" height="300">

In principle this can go on indefinitely, which lets the network condition its prediction on a long history. In our training code, ``sequence_len`` controls how many consecutive notes/events the model sees per training example, so during data preparation we slice the data into overlapping windows of that length. The trained RNN/LSTM *can* generate sequences longer than ``sequence_len``, but it has never been trained on inputs of that length, so prediction quality typically degrades.

This time, instead of describing notes by pitch and duration, we use a **uniform-time-step** representation, i.e. a piano roll representation: every event covers exactly one ``time_step`` (measured in quarter notes). A note is encoded as an ``X-NoteOn`` event — where ``X`` is the MIDI pitch — followed by zero or more ``NoteHold`` events (``_``). A rest is encoded analogously, as an ``r`` event followed by zero or more ``NoteHold`` events. For example:

```
65 _ _ _ 77 _ r _ _
```

means: the piece begins with MIDI pitch 65 (``65-NoteOn``), held for three more time steps (four in total); then MIDI pitch 77 (``77-NoteOn``), held for one more time step; finally, a rest of three time steps. We already showed how to convert a ``Stream`` into this *piano-roll-like* representation using the ``EventEncoder`` class.

### LSTMs

An **LSTM** (long short-term memory network) is an extension of the vanilla RNN that makes it much easier to learn from long sequences. Because a vanilla RNN compresses everything into a single hidden-state vector, information tends to get washed away over time, and the network has no built-in mechanism for deciding which past information is worth keeping.

The LSTM addresses this by introducing an additional **cell state** $\mathbf{c}_t$ — a kind of "long-term memory" — alongside the hidden state $\mathbf{h}_t$. Learned **gates** decide, at each time step, what to *erase* from $\mathbf{c}_t$, what to *write* into it, and what to *read out* as $\mathbf{h}_t$.

Don't worry too much about the diagram below. The key idea is that there are four weight matrices $W_f, W_i, W_g, W_o$ — all containing **learnable** parameters — that together decide, based on the current input $\mathbf{x}_t$ and the previous hidden state $\mathbf{h}_{t-1}$, what to forget and what to keep.

<img src="figs/lstm-cell.png" alt="" height="400">

+ **Forget gate** ($W_f$): decides which parts of the previous cell state $\mathbf{c}_{t-1}$ are no longer relevant and should be discarded.
+ **Input gate** ($W_i$), together with the **candidate cell state** ($W_g$): decide what new information to write into the cell state $\mathbf{c}_t$, preventing the accumulation of irrelevant information.
+ **Output gate** ($W_o$): controls how much of the updated cell state $\mathbf{c}_t$ is exposed as the new hidden state $\mathbf{h}_t$ — the "short-term memory" that is passed to the next time step and used to produce the prediction.

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 24.1:** Why would it be impossible to use the *piano roll like* representation for our simple neural network in ``9_6_ml_ffn_markov``?

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 24.1 Data Collection / Preparation

This time we not only use one piece of music but we use roughly 1000 melodies from folk songs. The ``.zip`` file ``data/deu_folk_songs.zip`` contains MIDI files of melodies. These files represent our **training data** for our *deep learning* attempts later.

In [ ]:
with zipfile.ZipFile('data/deu_folk_songs.zip', 'r') as zip_ref:
    zip_ref.extractall('data/deu_folk_songs/')

Let us unzip it. Then you can use the ``load_midi_files`` function to load all the MIDI files. It returns a list of ``Stream`` objects. You can and should specify a ``time_step`` to filter out files that can not be represented by the time step we want to use. We also transpose all the piece into the key of ``C major`` such that our model only has to learn one key!

In [ ]:
# this can take a some time!
import glob
time_step = 0.5
mid_files = glob.glob('data/deu_folk_songs/**/*.mid',recursive = True)
streams_folk_songs = load_midi_files(mid_files, time_step=time_step, transpose_to_major=True, max_files=100) 
print(f'load {len(streams_folk_songs)} files')

First we transform our `Score`s / `Stream`s into our piano roll representation.
Second, we prepare the piano roll representation to be *one-hot encoded* by transforming it into numbers ranging from $0$ to $m-1$.
This can be done by using the ``StringToIntEncoder`` appropriately.

In [ ]:
piano_roll_encoder = PianoRollEncoder(time_step=time_step)
piano_rolls, _ = piano_roll_encoder.encode_streams(streams_folk_songs)
print(f'piano rolls as strings: {piano_rolls[:3]}')

string_to_int = StringToIntEncoder(piano_rolls)
piano_rolls_int = string_to_int.encode_sequences(piano_rolls)
vocab_size = len(string_to_int)

print(f'piano roll as int: {piano_rolls_int[0]}')
print(f'vocab_size: {vocab_size}')

Next we have to generate our training dataset. However, this time the input is a sequence of events and the output is the next event!
The sequence lenght is our first (hyper-)parameter. To enable the network to be able to predict for shorter sequences we pad with ``TERM_SYMBOL``

In [ ]:
sequence_len = 64 # this is a hyper-paremter!

xs = []
ys = []
i_term_symbol = string_to_int.encode(TERM_SYMBOL)
for piano_roll in piano_rolls_int:
    padded = [i_term_symbol] * sequence_len + piano_roll + [i_term_symbol]
    for i in range(len(padded) - sequence_len):
        xs.append(padded[i:(i + sequence_len)])
        ys.append(padded[i + sequence_len])

print(xs[:3])
print(ys[:2])

In [ ]:
len(xs)

Instead of using *one-hot encoding* we let the network do the job.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split

X = torch.tensor(xs, dtype=torch.long)   # shape: (num_samples, sequence_len)
y = torch.tensor(ys, dtype=torch.long)   # shape: (num_samples,)

print(X.shape)
print(y.shape)

print(X[:3])
print(y[:2])

dataset = TensorDataset(X, y)

# 80 / 20 train / val split (deterministic via fixed seed)
val_frac = 0.2
val_size = int(len(dataset) * val_frac)
train_size = len(dataset) - val_size
generator = torch.Generator().manual_seed(42)
train_set, val_set = random_split(dataset, [train_size, val_size], generator=generator)

batch_size = 16
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=batch_size)

### 24.2 Model Definition

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hyperparameters!

vocab_size = len(string_to_int)   # size of our alphabet
input_dim = vocab_size            # can be different
hidden_dim = 16                   # can be different
layer_dim = 1                     # can be different
output_dim = vocab_size           # should not be different
dropout = 0.2                     # can be different

criterion = torch.nn.CrossEntropyLoss()

learning_rate = 0.001             # can be different
batch_size = 64                   # can be different
epochs = 20                       # can be different
eval_interval = 100               # can be different

In [ ]:
class LSTMModel(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, dropout=0.2):
        super(LSTMModel, self).__init__()

        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        
        self.embedding = torch.nn.Embedding(vocab_size, input_dim) # equivalent to one-hot with linear but without bias!
        self.lstm = torch.nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.dropout = torch.nn.Dropout(dropout)
        self.fc = torch.nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim, device=device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim, device=device)
        
        # x = B, T, C
        x = self.embedding(x)
        
        out, (ht, ct) = self.lstm(x, (h0, c0))
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out # B, C

In [ ]:
model = LSTMModel(input_dim, hidden_dim, layer_dim, output_dim, dropout).to(device)

for i in range(len(list(model.parameters()))):
    print(list(model.parameters())[i].shape)

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 24.2:** Look at the model summary above. Can you reason about the number of learnable parameters? 

🗣 **Hint:** Try to work through each ``torch.nn.Element`` and try to figure out how many learnable parameter there are. Look at the image of an ``torch.nn.LSTM`` we introduced in the beginning and consider that each of the ``4`` green boxes represents basically a layer of a neural network (a matrix and a bias term).

---

_Type your answer here, replacing this text._

In [ ]:
embed_params = vocab_size * input_dim
ih_params = 4 * vocab_size * hidden_dim + 4 * hidden_dim
hh_params = 4 * hidden_dim * hidden_dim + 4 * hidden_dim
dropout_params = 0
linear_params = hidden_dim * output_dim + output_dim

ih_params + hh_params + dropout_params + linear_params + embed_params

In [ ]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

<!-- END QUESTION -->

### 24.3 Model Training

Becaus the training can take some time, we already trained models using different hyper-paraemters. You can find them in the folder ``models/`` and instead of training, you can load them.

In [ ]:
import torch.nn as nn
from tqdm.auto import tqdm

loss_fn   = nn.CrossEntropyLoss() # computes the log_softmax
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# --- 3. Training loop -----------------------------------------------------
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()

    desc = f"Epoch {epoch}/{epochs} {'train' if train else 'val  '}"
    pbar = tqdm(loader, desc=desc, leave=False)

    with context:
        for xb, yb in pbar:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            #print(f'loss: {loss}')
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            correct    += (logits.argmax(dim=1) == yb).sum().item()
            total      += xb.size(0)

            pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.4f}")

    return total_loss / total, correct / total

for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)

    train_losses.append(tr_loss); train_accs.append(tr_acc)
    val_losses.append(va_loss);   val_accs.append(va_acc)

    print(f"Epoch {epoch:3d}/{epochs}  "
          f"train  loss {tr_loss:.4f}  acc {tr_acc:.4f}   |   "
          f"val  loss {va_loss:.4f}  acc {va_acc:.4f}")

Let's plot how the the training and valudation error behave.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(x=range(len(train_losses)), y=train_losses, linewidth=2.5, label='train', ax=axes[0])
sns.lineplot(x=range(len(val_losses)), y=val_losses, linewidth=2.5, label='val', ax=axes[0])

axes[0].set_title("Cross-entropy loss")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Loss")

plt.figure(figsize=(10, 5))
sns.lineplot(x=range(len(train_losses)), y=train_accs, linewidth=2.5, label='train', ax=axes[1])
sns.lineplot(x=range(len(val_losses)), y=val_accs, linewidth=2.5, label='val', ax=axes[1])

axes[1].set_title("Cross-entropy accuracy")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Accuracy")

plt.show()

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 24.3:** Find a good metaphor for the hyper-parameter ``hidde_state_dimension``. What might be the effect if we make it larger or smaller?

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 24.4 Save your Model (optional)

In [ ]:
torch.save(model.state_dict(), "models/lstm_model.pt")

### 24.5 Load a Model (optional)

In [ ]:
# 1) Modell mit identischer Architektur neu anlegen
model = LSTMModel(input_dim, hidden_dim, layer_dim, output_dim, dropout).to(device)

# 2) Gewichte einladen
model.load_state_dict(torch.load("models/lstm_model.pt", map_location=device))

# 3) In Inferenz-Modus schalten (schaltet Dropout aus, BN auf running stats)
model.eval()

### 24.6 Testing the Model

In [ ]:
my_melody = ['60', '_', '_', '50', '_', '_']
score = piano_roll_encoder.decode_stream(my_melody)
score.show('midi')

In [ ]:
# padding so the model "knows" that this is actually the start of a melody
my_melody = ([TERM_SYMBOL]*sequence_len + my_melody)[-sequence_len:]

In [ ]:
my_melody_int = string_to_int.encode_sequence(my_melody)
print(my_melody_int)
idx =  torch.tensor([my_melody_int], dtype=torch.long, device=device)
logits = model.forward(idx)[0]
probs = torch.softmax(logits, dim=-1)
probs

In [ ]:
symbol_int = torch.multinomial(probs, num_samples=1).item()
string_to_int.decode(symbol_int)

In [ ]:
def generate_melody(model: LSTMModel, sequence_len, seed, string_to_int: StringToIntEncoder, temperature: float=1.0, max_len=1_000) -> list[str]:

    # Pad with TERM_SYMBOL
    padded_seed = [TERM_SYMBOL]*sequence_len + seed
    
    # Tranform pedding + seed into numbers
    seed_int = string_to_int.encode_sequence(padded_seed)

    # Add the seed to the melody
    melody = seed[:]
    with torch.no_grad():
        while True:
            seed_int = seed_int[-sequence_len:]
            
            # Make prediction, note that we have a softmax-layer integrated into our model
            idx =  torch.tensor([seed_int], dtype=torch.long, device=device)
            logits = model.forward(idx)[0]

            if temperature <= 0:
                symbol_int = logits.argmax(dim=-1).item()
            else:
                probs = torch.softmax(logits / temperature, dim=-1)
                symbol_int = torch.multinomial(probs, num_samples=1).item()
                seed_int.append(symbol_int)
            
            symbol = string_to_int.decode(symbol_int)

            if symbol != TERM_SYMBOL:
                melody.append(symbol)
                if max_len <= len(melody):
                    break
            else:
                break
    return melody

In [ ]:
score_as_list = generate_melody(model, sequence_len=sequence_len, seed=['60', '_', '_', '50', '_', '_'], string_to_int=string_to_int, temperature=0.7, max_len=100)
print(score_as_list)
score = piano_roll_encoder.decode_stream(score_as_list)
score.show('midi')

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 24.4:** Explain the what the function ``generate_melody`` computes and how it works. Explain what the parameter ``temperature`` controls. You might want to generate multiple melodies with different ``temperature`` values. Listen to the result.

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 24.5:** We used all the data for training, neglecting any validation. Explain and think about:

1. What is the disadvantage of not validating our model? **Hint:** Think of the concept of *overfitting*.
2. What would happen if the model overfits its training data? Is this always bad in our context or do we might want this to happen?

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

